# Notebook 03 — HopfieldReplayBuffer + DQN

APBIO project, 2025/26.

This is where the hybrid system actually comes together. I subclass SB3's
`ReplayBuffer` so that instead of sampling transitions uniformly at random,
the buffer queries a Modern Hopfield memory with the **most recently added
state** and returns the transitions whose states are most similar to it.

DQN then performs its Bellman update on that Hopfield-selected mini-batch.

Plan for this notebook:

1. Define `HopfieldMemory` (same as notebook 02) and `HopfieldReplayBuffer`.
2. Run a short smoke test: one seed, 50k steps, hybrid DQN on CartPole.
3. Plot hybrid vs. baseline (already trained in notebook 01).

If hybrid trains without crashing and produces a sensible learning curve,
I move on to full multi-seed experiments in notebook 04.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/apbio_project')
(PROJECT_ROOT / 'logs').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

In [ ]:
%pip install -q 'gymnasium>=0.29' 'stable-baselines3>=2.3'

In [ ]:
import csv, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

from stable_baselines3 import DQN
from stable_baselines3.common.buffers import ReplayBuffer
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.type_aliases import ReplayBufferSamples


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 2. Hopfield memory (same as notebook 02)

Copy of the module from notebook 02 so this notebook is self-contained.

In [ ]:
class HopfieldMemory(nn.Module):
    def __init__(self, capacity, dim, beta=1.0):
        super().__init__()
        self.capacity = capacity
        self.dim = dim
        self.beta = beta
        self.register_buffer('patterns', torch.zeros(capacity, dim))
        self.register_buffer('size', torch.tensor(0, dtype=torch.long))

    def store(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        n = x.shape[0]
        s = int(self.size.item())
        idx = (torch.arange(n) + s) % self.capacity
        self.patterns[idx] = x.to(self.patterns.dtype)
        self.size.fill_(min(s + n, self.capacity))

    def retrieve_weights(self, query):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        s = int(self.size.item())
        if s == 0:
            raise RuntimeError('Hopfield memory is empty.')
        X = self.patterns[:s]
        scores = self.beta * query @ X.T
        return torch.softmax(scores, dim=-1), s

    def mean_energy(self, query):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        s = int(self.size.item())
        X = self.patterns[:s]
        lse = torch.logsumexp(self.beta * query @ X.T, dim=-1) / self.beta
        return (-lse + 0.5 * (query * query).sum(dim=-1)).mean().item()

## 3. HopfieldReplayBuffer

The design:

- `add()` stores the transition in SB3's normal arrays and also stores the
  observation in the Hopfield memory. I keep a copy of the most recent
  observation as `self._last_obs`.
- `sample(batch_size)` uses `self._last_obs` as a query, asks Hopfield for
  attention weights over all stored patterns, and draws a mini-batch of
  transition indices according to those weights (with replacement).
- `_get_samples(batch_inds)` is inherited as-is from `ReplayBuffer`.

A small detail: SB3 calls `sample()` in `DQN.train()`. Before enough
experiences have been collected (`learning_starts` steps), SB3 does not
call `sample()` at all, so I don't need to handle the empty-buffer case
there — but the check is free, so I leave it.

I also track the mean Hopfield energy on every `sample()` call; I will
plot it later to satisfy the 'internal dynamics' requirement of the
project spec.

In [ ]:
class HopfieldReplayBuffer(ReplayBuffer):
    """Replay buffer whose samples are drawn according to Hopfield attention
    weights conditioned on the most recently stored observation.
    """

    def __init__(self, buffer_size, observation_space, action_space,
                 device='auto', n_envs=1,
                 hopfield_beta=5.0, hopfield_capacity=None,
                 **kwargs):
        super().__init__(buffer_size, observation_space, action_space,
                         device=device, n_envs=n_envs, **kwargs)

        obs_dim = int(np.prod(observation_space.shape))
        cap = hopfield_capacity if hopfield_capacity is not None else buffer_size
        self.hopfield = HopfieldMemory(capacity=cap, dim=obs_dim, beta=hopfield_beta)

        self._last_obs = None                       # query state
        self.energy_trace = []                      # (timestep, mean_energy)
        self._sample_call_count = 0

    def add(self, obs, next_obs, action, reward, done, infos):
        super().add(obs, next_obs, action, reward, done, infos)

        # Store the (possibly batched) observation in the Hopfield memory.
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        if obs_t.dim() == 1:
            obs_t = obs_t.unsqueeze(0)
        self.hopfield.store(obs_t)

        # Remember the last observation for queries (use the last env if batched).
        self._last_obs = obs_t[-1].clone()

    def sample(self, batch_size, env=None):
        if self.size() == 0 or self._last_obs is None:
            return super().sample(batch_size, env=env)

        # Hopfield attention over all currently stored patterns.
        weights, n_stored = self.hopfield.retrieve_weights(self._last_obs)
        weights = weights.squeeze(0).detach().cpu().numpy()  # (n_stored,)

        # Normalise defensively in case of numerical drift.
        weights = weights / weights.sum()

        # Sample transition indices according to Hopfield weights, with replacement.
        # n_stored here equals the number of transitions added so far, which
        # matches SB3's internal buffer size as long as we add in the same order.
        upper = min(n_stored, self.size())
        weights = weights[:upper]
        weights = weights / weights.sum()
        batch_inds = np.random.choice(upper, size=batch_size, replace=True, p=weights)

        # Log mean energy every 50 sample() calls for the diagnostics plot.
        self._sample_call_count += 1
        if self._sample_call_count % 50 == 0:
            e = self.hopfield.mean_energy(self._last_obs)
            self.energy_trace.append((self._sample_call_count, e))

        # Delegate the actual tensor construction to the parent class.
        env_inds = np.zeros_like(batch_inds)   # single env
        return self._get_samples(batch_inds, env=env)

print('HopfieldReplayBuffer defined.')

## 4. Episode logger (same as notebook 01)

In [ ]:
class EpisodeLogger(BaseCallback):
    def __init__(self, csv_path):
        super().__init__()
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self._ep = 0
        with self.csv_path.open('w', newline='') as f:
            csv.writer(f).writerow(['episode', 'timestep', 'return', 'length'])

    def _on_step(self):
        finished = []
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self._ep += 1
                finished.append([
                    self._ep, self.num_timesteps,
                    float(info['episode']['r']), int(info['episode']['l']),
                ])
        if finished:
            with self.csv_path.open('a', newline='') as f:
                csv.writer(f).writerows(finished)
        return True

## 5. Smoke-test training function

Same hyperparameters as the baseline in notebook 01, plus two Hopfield
specific knobs (`beta` and `capacity`).

In [ ]:
TOTAL_TIMESTEPS = 50_000

def train_hybrid(seed, beta=5.0, hopfield_capacity=None,
                 total_timesteps=TOTAL_TIMESTEPS, verbose=1):
    set_seed(seed)
    env = Monitor(gym.make('CartPole-v1'))
    log_path = PROJECT_ROOT / 'logs' / f'hybrid_seed{seed}.csv'

    model = DQN(
        policy='MlpPolicy',
        env=env,
        learning_rate=2.3e-3,
        buffer_size=100_000,
        learning_starts=1_000,
        batch_size=64,
        tau=1.0,
        gamma=0.99,
        train_freq=256,
        gradient_steps=128,
        target_update_interval=10,
        exploration_fraction=0.16,
        exploration_final_eps=0.04,
        policy_kwargs=dict(net_arch=[256, 256]),
        replay_buffer_class=HopfieldReplayBuffer,
        replay_buffer_kwargs=dict(
            hopfield_beta=beta,
            hopfield_capacity=hopfield_capacity,
        ),
        seed=seed,
        verbose=verbose,
    )

    cb = EpisodeLogger(csv_path=str(log_path))
    t0 = time.time()
    model.learn(total_timesteps=total_timesteps, callback=cb, log_interval=10)
    elapsed = time.time() - t0

    # Save energy trace for diagnostics.
    energy_path = PROJECT_ROOT / 'logs' / f'hybrid_seed{seed}_energy.csv'
    with energy_path.open('w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['sample_call', 'mean_energy'])
        w.writerows(model.replay_buffer.energy_trace)

    return log_path, energy_path, elapsed

## 6. Run a single seed to check the integration works

In [ ]:
SEED = 0
log_path, energy_path, elapsed = train_hybrid(SEED, beta=5.0)
print(f'\nhybrid seed {SEED} done in {elapsed:.1f}s')
print('log   :', log_path)
print('energy:', energy_path)

## 7. Hybrid vs. baseline learning curves

The baseline CSVs were already produced in notebook 01 for seeds 0, 42,
123. Here I plot hybrid seed 0 against baseline seed 0 so the comparison
is paired.

In [ ]:
base_df = pd.read_csv(PROJECT_ROOT / 'logs' / f'baseline_seed{SEED}.csv')
hyb_df  = pd.read_csv(log_path)

base_df['smooth'] = base_df['return'].rolling(20, min_periods=1).mean()
hyb_df['smooth']  = hyb_df['return'].rolling(20, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(base_df['episode'], base_df['smooth'], label='baseline DQN', alpha=0.9)
ax.plot(hyb_df['episode'],  hyb_df['smooth'],  label='hybrid (Hopfield buffer)', alpha=0.9)
ax.axhline(500, color='grey', linestyle='--', linewidth=1, label='max (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Return (20-ep rolling mean)')
ax.set_title(f'Hybrid vs. baseline DQN on CartPole-v1 (seed {SEED})')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()

out = PROJECT_ROOT / 'figures' / f'hybrid_vs_baseline_seed{SEED}.png'
fig.savefig(out, dpi=150)
plt.show()
print('figure:', out)

## 8. Hopfield energy trace (internal dynamics)

In [ ]:
edf = pd.read_csv(energy_path)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(edf['sample_call'], edf['mean_energy'])
ax.set_xlabel('sample() call')
ax.set_ylabel('mean Hopfield energy of last_obs')
ax.set_title(f'Hopfield energy during training (seed {SEED})')
ax.grid(True, alpha=0.3)
fig.tight_layout()

out = PROJECT_ROOT / 'figures' / f'hopfield_energy_seed{SEED}.png'
fig.savefig(out, dpi=150)
plt.show()
print('figure:', out)

## 9. Quick summary

In [ ]:
def summarise(df, label):
    return dict(
        label=label,
        episodes=len(df),
        first_10_mean=round(df['return'].head(10).mean(), 1),
        last_10_mean =round(df['return'].tail(10).mean(), 1),
        max_return   =int(df['return'].max()),
    )

summary = pd.DataFrame([
    summarise(base_df, 'baseline'),
    summarise(hyb_df,  'hybrid'),
])
print(summary.to_string(index=False))

## What to look for

- If hybrid trains without errors and produces a rising learning curve,
  the integration is working.
- The hybrid does not need to beat the baseline on a single seed — seed
  variance is high. The proper comparison happens in notebook 04 with
  5 seeds each.
- What I really want to see is: no crashes, the Hopfield energy trace
  is non-trivial (not flat, not NaN), and the hybrid reaches reasonable
  returns (say >150 at some point).